In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import tifffile as tf
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm

from brisc.manuscript_analysis import rabies_cell_counting as rv_count
from brisc.manuscript_analysis import starter_cell_counting as sc_count
from brisc.manuscript_analysis import overview_image
from brisc.manuscript_analysis.utils import get_path


In [ ]:

DATA_ROOT = None
PROJECT = "rabies_barcoding"
MOUSE = "BRAC11816.2e"
USE_DOWNSAMPLED = False
arial_font_path = "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"

if arial_font_path is not None:
    try:
        arial_prop = fm.FontProperties(fname=arial_font_path)
        plt.rcParams["font.family"] = arial_prop.get_name()
        plt.rcParams.update({"mathtext.default": "regular"})
        fm.fontManager.addfont(arial_font_path)
    except FileNotFoundError:
        print("Arial font not found, proceeding with default font.")

matplotlib.rcParams["pdf.fonttype"] = 42

fontsize_dict = {"title": 7, "label": 8, "tick": 6, "legend": 6}
line_width = 1.2


In [ ]:
# Load Data

if USE_DOWNSAMPLED:
    mcherry_file = (
        get_path(PROJECT, data_root=DATA_ROOT)
        / MOUSE
        / "cellfinder_results_010/registration/downsampled_channel_0.tiff"
    )
    background_file = (
        get_path(PROJECT, data_root=DATA_ROOT)
        / MOUSE
        / "cellfinder_results_010/registration/downsampled.tiff"
    )
else:
    mcherry_file = (
        get_path(PROJECT, data_root=DATA_ROOT)
        / MOUSE
        / "z_project_70_100_ch2_median2x2x2.tif"
    )
    background_file = (
        get_path(PROJECT, data_root=DATA_ROOT)
        / MOUSE
        / "z_project_70_100_ch3_median2x2x2.tif"
    )

mcherry = tf.imread(mcherry_file)
background = tf.imread(background_file)

injection_centers = {
    "BRYC64.2h": np.array([567, 144, 864]),
    "BRYC64.2i": np.array([673, 205, 890]),
    "BRAC11816.2e": np.array([828, 65, 699]),
}

voxel_distances_sorted, cell_distances_sorted = rv_count.rv_cortical_cell_distances(
    inj_center=injection_centers[MOUSE],
    project=PROJECT,
    mouse=MOUSE,
    processed=DATA_ROOT,
    data_root=DATA_ROOT,
)
IMAGE_FILE = get_path(PROJECT, data_root=DATA_ROOT) / MOUSE  / 'BRAC11816.2e_slide3_slice20_MAX_projection_9slices_rotated90_median2x2x2.tif'
starter_img = tf.imread(IMAGE_FILE)
starter_img_metadata = None

In [ ]:
save_fig = False
save_path = get_path(PROJECT, data_root=DATA_ROOT)  / "home/shared/presentations/becalick_2025"


In [ ]:
# The new figsize is scaled down from (8.27, 11.69) based on the bounding box (0.62 x 0.39)
cm = 1 / 2.54
fig = plt.figure(figsize=(15*cm, 6*cm))

# ax_frame can be kept if you want a border, it will now tightly wrap the plots
ax_frame = fig.add_axes([0, 0, 1, 1])
ax_frame.set_xticks([])
ax_frame.set_yticks([])

# Recalculated coordinates to span from 0 to 1
ax_coronal_local_rabies1 = fig.add_axes([0, 0, 0.6, 1.0])
ax_coronal_local_rabies2 = fig.add_axes([0.34, 0, 0.3, 0.4])

if USE_DOWNSAMPLED:
    kwargs = dict(
        z_proj_size=200,
        vlim_mcherry=(1000, 6000),
        vlim_background=(0, 8000),
        zoom_row_lims=(-90, 340),
        zoom_col_lims=(-200, 290),
        use_downsampled=USE_DOWNSAMPLED
    )
else:
    kwargs = dict(
        z_proj_size=None,
        vlim_mcherry=(2000, 5000),
        vlim_background=(0, 5000),
        zoom_row_lims=(0, 3500),
        zoom_col_lims=(0, 5000),
        use_downsampled=USE_DOWNSAMPLED
    )

rv_count.plot_rv_coronal_slice(
    injection_centers[MOUSE],
    (ax_coronal_local_rabies1, ax_coronal_local_rabies2),
    mcherry,
    background,
    **kwargs
)

# confocal example
ax_starter_detection = fig.add_axes([0.55, 0.45, 0.5, 0.55])
if starter_img_metadata is None:
    pixel_size_um = 0.3107403
else:
    pixel_size_um=0.207

sc_count.plot_starter_confocal(ax_starter_detection, starter_img, None)
overview_image.add_scalebar(
    ax_starter_detection,
    downsample_factor=1,
    pixel_size_um=pixel_size_um,
    length_um=20,
    bar_height_px=7,
    margin_px=10,
    location="bottom left"
)

ax_presynaptic_density = fig.add_axes([0.75, 0.18, 0.20, 0.25])
im = rv_count.plot_rabies_density(
    inj_center=injection_centers[MOUSE],
    ax=ax_presynaptic_density,
    label_fontsize=fontsize_dict["label"],
    tick_fontsize=fontsize_dict["tick"],
    processed=get_path(PROJECT, DATA_ROOT).parent,
    linewidth=line_width,
    voxel_distances_sorted=voxel_distances_sorted,
    cell_distances_sorted=cell_distances_sorted,
)

if save_fig:
    fig.savefig(save_path / "preperstart_iv.pdf", bbox_inches='tight')
    print("Figure saved as coronal_local_rabies_panels.pdf")
